# Humira Patient Demand – Data Exploration

This notebook explores Medicare Part D and Medicaid annual drug spending datasets, extracts Humira data, builds a time series, and visualizes patient demand trends.

Dataset files expected in:

```
../data/Medicare_Part_D_Spending_by_Drug_2023.csv
../data/Medicaid_Spending_by_Drug_2023.csv
```

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

CURRENT_DIR = os.path.dirname(os.getcwd())  # notebooks/ → project root
DATA_DIR = os.path.join(CURRENT_DIR, "data")

MEDICARE_FILE = os.path.join(DATA_DIR, "Medicare_Part_D_Spending_by_Drug_2023.csv")
MEDICAID_FILE = os.path.join(DATA_DIR, "Medicaid_Spending_by_Drug_2023.csv")

TARGET_DRUG = "HUMIRA"

## Load Data

In [ ]:
medicare = pd.read_csv(MEDICARE_FILE)
medicaid = pd.read_csv(MEDICAID_FILE)

medicare.head(), medicaid.head()

## Filter Humira

In [ ]:
def filter_humira(df):
    df = df.copy()
    df["Brnd_Name_upper"] = df["Brnd_Name"].astype(str).str.upper()
    return df[df["Brnd_Name_upper"] == TARGET_DRUG]

medicare_h = filter_humira(medicare)
medicaid_h = filter_humira(medicaid)

medicare_h.head(), medicaid_h.head()

## Convert Wide → Long Time Series (2019–2023)

In [ ]:
def wide_to_long_claims(df, source_label):
    records = []
    for year in range(2019, 2024):
        col = f"Tot_Clms_{year}"
        if col in df.columns:
            total_claims = df[col].sum()
            records.append({"year": year, "total_claims": total_claims, "source": source_label})
    return pd.DataFrame(records)

medicare_ts = wide_to_long_claims(medicare_h, "Medicare Part D")
medicaid_ts = wide_to_long_claims(medicaid_h, "Medicaid")

combined_ts = (
    pd.concat([medicare_ts, medicaid_ts], ignore_index=True)
    .groupby("year", as_index=False)["total_claims"].sum()
    .rename(columns={"total_claims": "total_claims_all"})
)

combined_ts

## Plot Historical Patient Demand

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(combined_ts["year"], combined_ts["total_claims_all"], marker="o")
plt.title("Humira – Total Claims (Medicare + Medicaid)")
plt.xlabel("Year")
plt.ylabel("Total Claims")
plt.grid(True)
plt.show()

# Prophet Forecasting

In [ ]:
from prophet import Prophet

prophet_df = combined_ts.rename(columns={"year": "ds", "total_claims_all": "y"})
prophet_df["ds"] = pd.to_datetime(prophet_df["ds"].astype(str) + "-01-01")

m = Prophet(yearly_seasonality=False)
m.fit(prophet_df)

future = m.make_future_dataframe(periods=5, freq="YE")
forecast = m.predict(future)

forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

In [ ]:
fig = m.plot(forecast)
plt.title("Humira – Prophet Forecast")
plt.show()

# XGBoost Forecasting

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

X = combined_ts[["year"]]
y = combined_ts["total_claims_all"]

X_train, X_test = X.iloc[:-1], X.iloc[-1:]
y_train, y_test = y.iloc[:-1], y.iloc[-1:]

model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=3)
model.fit(X_train, y_train)

y_pred_test = model.predict(X_test)
rmse = (mean_squared_error(y_test, y_pred_test)) ** 0.5
rmse

In [ ]:
future_years = pd.DataFrame({"year": range(combined_ts["year"].max() + 1, combined_ts["year"].max() + 6)})
future_pred = model.predict(future_years)

plt.figure(figsize=(10, 6))
plt.plot(combined_ts["year"], combined_ts["total_claims_all"], marker="o", label="Historical")
plt.plot(future_years["year"], future_pred, marker="x", linestyle="--", label="Forecast")
plt.title("Humira – XGBoost Forecast")
plt.xlabel("Year")
plt.ylabel("Total Claims")
plt.legend()
plt.grid(True)
plt.show()